# Projet Final : Biais et Équité dans la Classification d'Images Médicales
## NIH Chest X-ray — ResNet18

### Trinôme : Othmane Nammous, Tharushan Uthayakumar, Rémi Malapert

---

## 0/- Introduction

Ce projet étudie l'équité en intelligence artificielle appliquée à l'imagerie médicale à partir d'un sous-ensemble du dataset NIH Chest X-ray 14. L'objectif est d'identifier les biais présents dans les métadonnées puis de comparer plusieurs stratégies de mitigation sur un modèle de classification d'images.

Le ResNet18 prédit si un patient est malade ou sain à partir de l'image seule. Il peut pourtant reproduire certains biais associés au genre, à la position de vue ou à l'âge si ces variables sont déséquilibrées dans les données.

### 0.1 Objectif du notebook

Le notebook suit une logique progressive proche du mi-projet:
1. chargement et préparation des données,
2. analyse descriptive et repérage des biais,
3. mesures de fairness sur les données,
4. pré-processing par pondération,
5. comparaison finale des approches.

L'analyse reste volontairement concise: on conserve les graphiques et tableaux qui montrent un déséquilibre utile à l'interprétation, sans multiplier les visualisations redondantes.

# 0/- Introduction (explication du cas d'usage et de l'objectif) - [à adapter]

## 1) Contexte et cas d'usage

### 1.1 Le diagnostic médical assisté par IA
Les radiographies thoraciques (chest X-rays) constituent l'un des examens d'imagerie médicale les plus fréquents au monde. Bien qu'elles soient rapides et peu coûteuses, leur interprétation clinique reste complexe et sujette à erreurs. L'intelligence artificielle offre aujourd'hui un potentiel considérable pour assister les radiologues via des systèmes de CAD (Computer-Aided Diagnosis).

Cependant, le déploiement de ces systèmes soulève un enjeu éthique majeur : l'équité. Les algorithmes d'apprentissage supervisé reproduisent mécaniquement les biais présents dans leurs données d'entraînement. Si un dataset contient des déséquilibres historiques (par exemple, une sous-représentation des femmes ou des diagnostics moins précis selon l'âge), l'IA risque d'amplifier ces inégalités, conduisant à des pertes de chance pour certaines populations de patients.

### 1.2 Présentation du dataset
Nous travaillons sur un sous-ensemble du NIH Chest X-ray Dataset (Wang et al., 2017).
Pour ce projet, nous analysons un jeu de données contenant environ 54 000 images. Chaque ligne correspond aux métadonnées d'une radiographie (identifiants (Patient ID, numéro de suivi), données démographiques (Patient Age, Patient Gender), données cliniques (Finding Labels), et données techniques (View Position)

Ce projet se concentre exclusivement sur l'audit des métadonnées structurées (cf fichier CSV) et n'inclut pas le traitement des images brutes.

## 2) Objectifs et méthodologie
L'objectif de ce notebook est d'auditer ce dataset médical pour détecter, quantifier et corriger les biais potentiels de représentation avant tout entraînement de modèle. Notre démarche s'articule autour de quatre axes :

1. **Analyse exploratoire et nettoyage** :
    - Traitement des valeurs aberrantes (notamment filtrage des âges > 100 ans)
    - Encodage des pathologies (One-Hot Encoding) pour permettre l'analyse statistique
    - Visualisation des distributions (âge, genre, pathologies, position de vue) pour identifier les déséquilibres manifestes

2. **Mesure de la Fairness** :
Nous quantifierons les biais en utilisant deux métriques standardisées :
    - **Statistical Parity Difference (SPD)** : mesure la différence de taux de sélection entre groupes (seuil : |SPD| < 0.1)
    - **Disparate Impact Ratio (DIR)** : mesure le rapport des taux de sélection (seuil : DIR entre 0.8 et 1.25)

    L'analyse porte sur :
    - L'attribut sensible : le genre (Homme/Femme)
    - Deux variables cibles : la présence de toute pathologie (`has_pathology`) et la présence spécifique d'Infiltration (`has_infiltration`)
    - Le groupe privilégié : les hommes (groupe majoritaire dans l'historique des données)
    - Une analyse par pathologie individuelle et une analyse au niveau patient (vs. niveau image)

3. **Mitigation (preprocessing)** :
Nous appliquerons deux méthodes de correction des biais directement sur les données, via le framework AIF360 :
    - **Reweighing** : attribue un poids différent à chaque instance pour neutraliser le lien statistique entre le genre et le diagnostic, garantissant un apprentissage plus équitable. Appliqué sur les deux cibles (globale et Infiltration).
    - **Disparate Impact Remover (DIR)** : modifie les valeurs des features (sans toucher aux labels) pour réduire la corrélation entre l'attribut sensible et les caractéristiques, tout en préservant le rang au sein de chaque groupe.

4. **Comparaison et conclusion** :
    - Comparaison quantitative des résultats avant/après mitigation (DI, SPD)
    - Analyse critique des forces et limites de chaque méthode

In [88]:
# === Imports ===
import os
import random
import shutil
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torch

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", append=True, category=UserWarning)

# Métriques de fairness (AIF360 sklearn API)
from aif360.sklearn.metrics import (
    statistical_parity_difference,
    disparate_impact_ratio,
    equal_opportunity_difference,
    average_odds_difference,
    base_rate,
    smoothed_edf,
    df_bias_amplification,
    conditional_demographic_disparity,
)
#from sklearn.metrics import balanced_accuracy_score, accuracy_score

# AIF360 pour le post-processing
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric
from aif360.algorithms.postprocessing.reject_option_classification import RejectOptionClassification
from aif360.algorithms.postprocessing.calibrated_eq_odds_postprocessing import CalibratedEqOddsPostprocessing

# Entraînement du modèle
#from train_classifieur import train_classifier, pred_classifier
#import torch
from train_classifieur import (
    ChestXRayClassifier,
    pred_classifier,
    preds_todf,
    train_classifier,
    transforms_valid,
)

In [89]:
import os
print(os.getcwd())

c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI


In [90]:
# === Configuration ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device utilisé : {device}")
if device.type == 'cuda':
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"Mémoire allouée : {round(torch.cuda.memory_allocated(0)/1024**3,1)} GB")

# Chemins portables pour le workspace courant
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "Data")
EXPE_DIR = os.path.join(BASE_DIR, "expe_log")
WORK_DIR = EXPE_DIR
CSV_PATH = os.path.join(DATA_DIR, "metadata.csv")

# Mode simple: on réutilise directement le dataset train/valid existant
SPLIT_CSV_PATH = CSV_PATH
SPLIT_DATA_DIR = DATA_DIR

os.makedirs(EXPE_DIR, exist_ok=True)

# Rendu final: garder False.
# Debug local rapide: mettre True pour 1 époque.
SMOKE_TEST = True
MAX_EPOCHS = 1 if SMOKE_TEST else 25
print(f"Mode smoke test: {SMOKE_TEST} | max_epochs={MAX_EPOCHS}")

Device utilisé : cpu
Mode smoke test: True | max_epochs=1


In [113]:
def add_binary_columns(df):
    df_out = df.copy()
    df_out['label_binary'] = (df_out['label'] == 'malade').astype(int)
    df_out['Gender_Binary'] = (df_out['Patient Gender'] == 'M').astype(int)
    df_out['Position_Binary'] = (df_out['View Position'] == 'PA').astype(int)
    return df_out


def prepare_experiment_data():
    """Charge le CSV existant et conserve le split train/valid déjà présent."""
    df_split = add_binary_columns(pd.read_csv(CSV_PATH))

    if 'train_valid' not in df_split.columns:
        raise ValueError(
            "La colonne 'train_valid' est absente de metadata.csv. "
            "Le mode simple attend un split train/valid déjà défini."
        )

    # Normalisation des libellés de split pour éviter les ambiguïtés
    df_split['train_valid'] = (
        df_split['train_valid']
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({'val': 'valid', 'validation': 'valid'})
    )

    allowed = {'train', 'valid'}
    unknown = sorted(set(df_split['train_valid'].unique()) - allowed)
    if unknown:
        raise ValueError(f"Valeurs de split non supportées en mode simple: {unknown}")

    df_split['train_valid'] = pd.Categorical(
        df_split['train_valid'],
        categories=['train', 'valid'],
        ordered=True,
    )

    df_split.to_csv(SPLIT_CSV_PATH, index=False)
    return df_split.copy(), df_split


def predict_partition(csv_path, split_root, partition, ckpt_path, preds_col='preds'):
    """Prédit une partition ImageFolder (train/valid) et prépare les colonnes utiles à l'évaluation."""
    train_dataset = ImageFolder(os.path.join(split_root, 'train'), transform=transforms_valid)
    label_decoder = {v: k for k, v in train_dataset.class_to_idx.items()}
    dataset = ImageFolder(os.path.join(split_root, partition), transform=transforms_valid)

    model = ChestXRayClassifier(adamax=True, cosine=True, nb_classes=len(label_decoder))
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu')['state_dict'])
    model.eval()

    df_partition = pd.read_csv(csv_path)
    df_partition = df_partition[df_partition['train_valid'] == partition].copy()
    df_partition[preds_col] = None
    df_partition = preds_todf(df_partition, dataset, label_decoder, model, preds_col)

    # Colonnes binaires standardisées pour les cellules d'évaluation
    truth_col = 'labels' if 'labels' in df_partition.columns else 'label'
    df_partition['label_binary'] = (df_partition[truth_col] == 'malade').astype(int)
    df_partition['preds_binary'] = (df_partition[preds_col] == 'malade').astype(int)
    if 'Gender_Binary' not in df_partition.columns:
        df_partition['Gender_Binary'] = (df_partition['Patient Gender'] == 'M').astype(int)
    if 'Position_Binary' not in df_partition.columns:
        df_partition['Position_Binary'] = (df_partition['View Position'] == 'PA').astype(int)

    return df_partition


df_raw, df_split = prepare_experiment_data()
df = df_split
print(f"Split train/valid prêt: {df_split['train_valid'].value_counts().to_dict()}")
print(f"CSV utilisé: {SPLIT_CSV_PATH}")

Split train/valid prêt: {'train': 3976, 'valid': 1246}
CSV utilisé: c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\Data\metadata.csv


In [92]:
df.describe()


,Follow-up #,Patient ID,Patient Age,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,WEIGHTS,label_binary,Gender_Binary,Position_Binary
count,5222.000000,5222.000000,5222.000000,5222.000000,5222.000000,5222.000000,5222.000000,0.0,5222.0,5222.000000,5222.000000,5222.000000
mean,8.023746,14087.229223,44.992149,2639.062428,2489.886059,0.155783,0.155783,NaN,1.0,0.452700,0.595557,0.612792
std,12.636620,8710.367565,18.289353,353.373602,394.052927,0.016832,0.016832,NaN,0.0,0.497805,0.490831,0.487158
min,0.000000,8.000000,2.000000,1490.000000,1297.000000,0.139000,0.139000,NaN,1.0,0.000000,0.000000,0.000000
25%,0.000000,6674.000000,30.000000,2500.000000,2048.000000,0.143000,0.143000,NaN,1.0,0.000000,0.000000,0.000000
50%,3.000000,13557.000000,46.000000,2530.000000,2544.000000,0.143000,0.143000,NaN,1.0,0.000000,1.000000,1.000000
75%,10.000000,20725.000000,58.000000,2992.000000,2991.000000,0.168000,0.168000,NaN,1.0,1.000000,1.000000,1.000000
max,76.000000,30795.000000,412.000000,3056.000000,3056.000000,0.194314,0.194314,NaN,1.0,1.000000,1.000000,1.000000


## I/- Analyse rapide des données

Dans cette partie nous allons étudier la distribution de chaque caractéristique et sa relation avec la variable cible, afin de mettre en évidence d'éventuels biais dans les données. [à adapter]

### 1.1 Préparation et nettoyage des données

En observant le `describe()` ci-dessus, on remarque que l'âge maximum est de **412 ans**, ce qui est impossible. Il faut traiter ce type de valeur aberrante afin de ne pas fausser les moyennes, les graphiques et les métriques de fairness.

Nous filtrons donc les patients dont l'âge est supérieur à 100 ans.

## je sais pas si c'est ici, ni s'il faut le faire:

Pour notre analyse de fairness, nous avons besoin d'une variable cible binaire. Nous créons la colonne **`has_pathology`** :
- **1** si le patient présente au moins une pathologie (Finding Labels != "No Finding")
- **0** si aucun diagnostic n'est détecté (Finding Labels = "No Finding")


Pour affiner notre analyse de biais, nous choisissons une pathologie spécifique comme variable cible en complément de la variable binaire `has_pathology`. La pathologie **Infiltration** est l'une des plus fréquentes dans le dataset, ce qui assure une taille d'échantillon suffisante pour des calculs de métriques de fairness fiables.

Faire cela à deux intérêts:
- La variable `has_pathology` agrège toutes les pathologies et peut **masquer des biais** propres à certaines maladies.
- En se concentrant sur **Infiltration**, on peut observer si le biais de genre est plus marqué pour une pathologie spécifique que pour l'ensemble des diagnostics. C'est une approche intéressante pour l'audit de fairness : analyser le biais plus finement.

L'attribut sensible principal est le **genre du patient**. Nous le codons en binaire :
- **1** pour les hommes (M), considéré comme le groupe privilégié (car majoritaire dans le dataset)
- **0** pour les femmes (F), considéré comme le groupe non-privilégié

Ce choix de groupe privilégié / non-privilégié est purement conventionnel et permet d'appliquer les métriques de fairness vues en cours (disparate impact, statistical parity, etc.). Le fait de désigner un groupe comme "privilégié" ne signifie pas qu'il est effectivement avantagé, c'est justement ce que nous allons déterminer.

Le champ `Finding_Labels` peut contenir plusieurs pathologies séparées par le caractère `|` (exemple vu dans le head plus haut : "Hernia|Infiltration"). Nous créons donc une colonne binaire par pathologie avec le **one-hot encoding**, afin d'analyser la distribution de chaque pathologie par genre.

In [93]:
# Extraire les noms des colonnes du CSV metadata
column_names = ['Image Index', 'Finding Labels', 'Follow-up #', 
                     'Patient ID', 'Patient Age', 'Patient Gender', 
                     'View Position', 'OriginalImage[Width, Height]', 
                     'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11', 
                     'train_valid', 'label', 'WEIGHTS', 'label_binary', 
                     'Gender_Binary', 'Position_Binary']

df = pd.read_csv(
    'Data/metadata.csv', # à modifier selon le chemin d'acces
    names=column_names,
    skiprows=1,
    header=None)

print(f"Nombre d'images : {len(df)}")
print(f"Nombre de patients uniques : {df['Patient ID'].nunique()}")
df.head(10)

Nombre d'images : 5222
Nombre de patients uniques : 88


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,"OriginalImage[Width, Height]",OriginalImagePixelSpacing[x,y],Unnamed: 11,train_valid,label,WEIGHTS,label_binary,Gender_Binary,Position_Binary
00000008_000.png,Cardiomegaly,0,8,69,F,PA,2048,2500,0.171,0.171,NaN,train,malade,1,1,0,1
00000008_001.png,No Finding,1,8,70,F,PA,2048,2500,0.171,0.171,NaN,train,sain,1,0,0,1
00000008_002.png,Nodule,2,8,73,F,PA,2048,2500,0.168,0.168,NaN,train,malade,1,1,0,1
00000011_000.png,Effusion,0,11,75,M,PA,2638,2449,0.143,0.143,NaN,train,malade,1,1,1,1
00000011_001.png,No Finding,1,11,75,M,PA,2500,2048,0.168,0.168,NaN,train,sain,1,0,1,1
00000011_002.png,No Finding,2,11,75,M,PA,2714,2781,0.143,0.143,NaN,train,sain,1,0,1,1
00000011_003.png,No Finding,3,11,75,M,PA,2500,2048,0.168,0.168,NaN,train,sain,1,0,1,1
00000011_004.png,No Finding,4,11,75,M,PA,2500,2048,0.168,0.168,NaN,train,sain,1,0,1,1
00000011_005.png,Infiltration,5,11,75,M,AP,2500,2048,0.168,0.168,NaN,train,malade,1,1,1,0
00000011_006.png,Atelectasis,6,11,75,M,PA,2992,2991,0.143,0.143,NaN,train,malade,1,1,1,1


In [94]:
# === Nettoyage et aperçu des données (style mi-projet) ===
df = df_raw.copy()

# Filtrage des âges aberrants dès la préparation
df_before_age_filter = df.copy()
df = df[df['Patient Age'] < 100].copy()
n_removed_age = len(df_before_age_filter) - len(df)

split_counts = df_split['train_valid'].value_counts().reindex(['train', 'valid']).fillna(0).astype(int)
label_counts = df['label'].value_counts().reindex(['sain', 'malade']).fillna(0).astype(int)
gender_counts = df['Patient Gender'].value_counts().reindex(['F', 'M']).fillna(0).astype(int)
position_counts = df['View Position'].value_counts().reindex(['AP', 'PA']).fillna(0).astype(int)

# Palette proche des defaults Plotly utilises dans le mi-projet
plotly_colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']

def add_count_bar(fig, series, row, col, color):
    total = series.sum()
    df_plot = pd.DataFrame({'cat': series.index.astype(str), 'count': series.values})
    df_plot['pct'] = 100 * df_plot['count'] / total if total > 0 else 0.0
    df_plot['text'] = df_plot['count'].astype(str) + ' (' + df_plot['pct'].round(1).astype(str) + '%)'
    fig.add_trace(
        go.Bar(
            x=df_plot['count'],
            y=df_plot['cat'],
            orientation='h',
            marker_color=color,
            text=df_plot['text'],
            textposition='outside',
            cliponaxis=False,
            showlegend=False,
            hovertemplate='%{y}: %{x}<extra></extra>',
        ),
        row=row,
        col=col,
    )

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Répartition train / valid', 'Labels (sain vs malade)',
        'Genre', 'Position de vue'
    ],
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

add_count_bar(fig, split_counts, 1, 1, plotly_colors[0])
add_count_bar(fig, label_counts, 1, 2, plotly_colors[1])
add_count_bar(fig, gender_counts, 2, 1, plotly_colors[2])
add_count_bar(fig, position_counts, 2, 2, plotly_colors[3])

fig.update_xaxes(title_text="Nombre d'images")
fig.update_yaxes(title_text='')
fig.update_layout(
    height=760,
    template='plotly',
    title_text='Aperçu du dataset après nettoyage',
    margin=dict(l=40, r=40, t=90, b=40),
)
fig.show()

age_fig = px.histogram(
    df,
    x='Patient Age',
    color='Patient Gender',
    nbins=30,
    opacity=0.7,
    marginal='box',
    barmode='overlay',
    title="Distribution de l'âge après filtrage des valeurs aberrantes",
    labels={'Patient Age': 'Âge', 'Patient Gender': 'Genre'},
)
age_fig.update_layout(template='plotly', height=500)
age_fig.show()

summary_df = pd.DataFrame({
    'Mesure': [
        'Images avant filtrage',
        'Images après filtrage',
        'Images retirées',
        'Taux de maladie',
        "Proportion d'hommes",
        'Proportion de PA',
        'Âge moyen',
        'Âge médian',
    ],
    'Valeur': [
        len(df_before_age_filter),
        len(df),
        n_removed_age,
        f"{(df['label'] == 'malade').mean():.2%}",
        f"{(df['Patient Gender'] == 'M').mean():.2%}",
        f"{(df['View Position'] == 'PA').mean():.2%}",
        f"{df['Patient Age'].mean():.1f}",
        f"{df['Patient Age'].median():.1f}",
    ]
})
summary_df

,Mesure,Valeur
0,Images avant filtrage,5222
1,Images après filtrage,5219
2,Images retirées,3
3,Taux de maladie,45.28%
4,Proportion d'hommes,59.55%
5,Proportion de PA,61.26%
6,Âge moyen,44.9
7,Âge médian,46.0


**Analyse des graphiques** [à adapter]
- **Analyse de la distribution d'âge par genre :**

On observe une superposition globale des distributions quasi-gaussienne centrée autour de 45-55 ans pour les deux genres. L'age n'est donc pas un facteur de confusion.


Par ailleurs on observe une chute des effectifs après 60 ans. Cependant cette baisse est démographique : elle reflète la structure de la population générale (moins d'individus très âgés en vie) plutôt qu'une baisse du risque pathologique.
Ainsi bien que les personnes âgées soient biologiquement plus sujettes aux pathologies, elles représentent un faible volume de données. Cela peut par la suite poser un risque de sous-ajustement du modèle qui verra peu d'exemples de patients de plus de 80 ans et risque d'être moins performant pour cette catégorie, créant une inégalité de traitement liée à l'âge.

### 1.2 Analyse descriptive et biais initiaux

In [95]:
# === Analyse descriptive visuelle (style mi-projet) ===

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Label par genre', 'Label par position',
        'Âge par genre', 'Prévalence par sous-groupe'
    ],
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'histogram'}, {'type': 'bar'}]]
)

ct_gender = pd.crosstab(df['Patient Gender'], df['label'], normalize='index') * 100
fig.add_trace(
    go.Bar(
        x=ct_gender.index.tolist(),
        y=ct_gender['malade'].tolist(),
        text=ct_gender['malade'].round(1).astype(str) + '%',
        textposition='outside',
        name='Malade',
    ),
    row=1,
    col=1,
 )
fig.add_trace(
    go.Bar(
        x=ct_gender.index.tolist(),
        y=ct_gender['sain'].tolist(),
        text=ct_gender['sain'].round(1).astype(str) + '%',
        textposition='outside',
        name='Sain',
    ),
    row=1,
    col=1,
 )

ct_pos = pd.crosstab(df['View Position'], df['label'], normalize='index') * 100
fig.add_trace(
    go.Bar(
        x=ct_pos.index.tolist(),
        y=ct_pos['malade'].tolist(),
        text=ct_pos['malade'].round(1).astype(str) + '%',
        textposition='outside',
        name='Malade',
        showlegend=False,
    ),
    row=1,
    col=2,
 )
fig.add_trace(
    go.Bar(
        x=ct_pos.index.tolist(),
        y=ct_pos['sain'].tolist(),
        text=ct_pos['sain'].round(1).astype(str) + '%',
        textposition='outside',
        name='Sain',
        showlegend=False,
    ),
    row=1,
    col=2,
 )

for gender in ['M', 'F']:
    ages = df.loc[df['Patient Gender'] == gender, 'Patient Age']
    fig.add_trace(
        go.Histogram(
            x=ages,
            name=gender,
            opacity=0.7,
            nbinsx=30,
        ),
        row=2,
        col=1,
    )

ct_cross = df.groupby(['Patient Gender', 'View Position'])['label_binary'].mean().reset_index()
ct_cross['group'] = ct_cross['Patient Gender'] + ' / ' + ct_cross['View Position']
ct_cross['prevalence_pct'] = (100 * ct_cross['label_binary']).round(1)
fig.add_trace(
    go.Bar(
        x=ct_cross['group'].tolist(),
        y=ct_cross['label_binary'].tolist(),
        text=ct_cross['prevalence_pct'].astype(str) + '%',
        textposition='outside',
        showlegend=False,
    ),
    row=2,
    col=2,
 )

fig.update_yaxes(title_text='Pourcentage (%)', range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text='Pourcentage (%)', range=[0, 100], row=1, col=2)
fig.update_yaxes(title_text='Prévalence', tickformat='.0%', row=2, col=2)
fig.update_layout(
    barmode='stack',
    height=780,
    template='plotly',
    title_text='Lecture rapide des déséquilibres structurants',
)
fig.show()

stats = pd.DataFrame({
    'Indicateur': ['Taux de maladie', 'Hommes', 'PA', 'Âge moyen', 'Âge médian'],
    'Valeur': [
        f"{(df['label'] == 'malade').mean():.2%}",
        f"{(df['Patient Gender'] == 'M').mean():.2%}",
        f"{(df['View Position'] == 'PA').mean():.2%}",
        f"{df['Patient Age'].mean():.1f}",
        f"{df['Patient Age'].median():.1f}",
    ]
})
stats

,Indicateur,Valeur
0,Taux de maladie,45.28%
1,Hommes,59.55%
2,PA,61.26%
3,Âge moyen,44.9
4,Âge médian,46.0


- On l'avait déjà vu tout à l'heure et c'est encore le cas ici: la répartition des pathologies semble relativement équilibrée entre hommes et femmes.

D'un autre côté on observe une répartition très différente des cas pathologiques selon la position de prise de vue :
La vue position PA présente une faible proportion de malades (41%) par rapport au non-malades (59%). 
On constate une Cette sur-représentation des cas pathologiques dans la catégorie AP est également présente en moindre mesure.


La répartition AP/PA est très similaire entre les deux genres (écart d'environ 3 points de pourcentage seulement). Cela signifie que la position de vue n'est pas un facteur confondant majeur dans ce dataset, les différences de taux de pathologie observées entre hommes et femmes ne peuvent pas être principalement attribuées à une différence de proportion AP/PA. 

Les deux genres ont majoritairement des radiographies PA (~60%).

In [96]:
# === Métriques de fairness sur les données brutes (labels réels) ===

spd_g = statistical_parity_difference(
    y_true=df['label_binary'], prot_attr=df['Gender_Binary'], pos_label=1)
dir_g = disparate_impact_ratio(
    y_true=df['label_binary'], prot_attr=df['Gender_Binary'], pos_label=1)

spd_p = statistical_parity_difference(
    y_true=df['label_binary'], prot_attr=df['Position_Binary'], pos_label=1)
dir_p = disparate_impact_ratio(
    y_true=df['label_binary'], prot_attr=df['Position_Binary'], pos_label=1)

fairness_summary = pd.DataFrame({
    'Attribut': ['Genre', 'Position'],
    'SPD': [spd_g, spd_p],
    'DIR': [dir_g, dir_p],
    'Statut': [
        'OK' if 0.8 <= dir_g <= 1.25 else 'Biais',
        'OK' if 0.8 <= dir_p <= 1.25 else 'Biais',
    ]
})

fairness_summary['Lecture rapide'] = [
    'SPD proche de 0, DIR proche de 1',
    'SPD proche de 0, DIR proche de 1',
]

display(
    fairness_summary.style
    .format({'SPD': '{:+.4f}', 'DIR': '{:.4f}'})
    .hide(axis='index')
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center')]},
        {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-weight', 'bold')]},
    ])
    .set_caption('Métriques de fairness sur les labels bruts')
)

print(f"Genre    -> SPD = {spd_g:+.4f} | DIR = {dir_g:.4f} | {'OK' if 0.8 <= dir_g <= 1.25 else 'Biais'}")
print(f"Position -> SPD = {spd_p:+.4f} | DIR = {dir_p:.4f} | {'OK' if 0.8 <= dir_p <= 1.25 else 'Biais'}")

Attribut,SPD,DIR,Statut,Lecture rapide
Genre,-0.0364,0.9221,OK,"SPD proche de 0, DIR proche de 1"
Position,+0.0908,1.2175,OK,"SPD proche de 0, DIR proche de 1"


Genre    -> SPD = -0.0364 | DIR = 0.9221 | OK
Position -> SPD = +0.0908 | DIR = 1.2175 | OK


### 1.3 Outils de mesure (performance et fairness)

[à adapter] 

Dans cette partie nous allons calculer les métriques de fairness directement sur les données. Comme nous ne disposons pas de prédictions de modèle, nous évaluons ici les biais inhérents aux données elles-mêmes.

Les métriques clés que nous calculons sont :

- **Base rate** : taux de positifs (ici, taux de pathologie) dans l'ensemble du dataset.

- **Statistical Parity Difference (SPD)** : mesure la différence absolue de taux de positifs entre les deux groupes. La formule est :

$$SPD = P(\hat{Y}=1 \mid S=\text{non-privilégié}) - P(\hat{Y}=1 \mid S=\text{privilégié})$$

  Dans notre cas : $SPD = P(\text{pathologie} \mid \text{Femme}) - P(\text{pathologie} \mid \text{Homme})$
  
  - $SPD = 0$ → parité parfaite.
  - $SPD < 0$ → les femmes ont un taux de pathologie **inférieur** aux hommes.
  - $SPD > 0$ → les femmes ont un taux de pathologie **supérieur** aux hommes.

- **Disparate Impact Ratio (DIR)** : ratio des taux de positifs entre les groupes :

$$DIR = \frac{P(\hat{Y}=1 \mid S=\text{non-privilégié})}{P(\hat{Y}=1 \mid S=\text{privilégié})} = \frac{P(\text{pathologie} \mid \text{Femme})}{P(\text{pathologie} \mid \text{Homme})}$$

  - $DIR = 1$ → parité parfaite.
  - La **règle des 80%**  considère qu'un $DIR < 0.8$ indique un impact disproportionné.


In [97]:
def get_metrics(y_true, y_pred=None, prot_attr=None, priv_group=1, pos_label=1, sample_weight=None):
    """
    Calcule un ensemble complet de métriques de fairness et de performance.
    Adapté du TD3/TD4 (aif360 sklearn API).

    Args:
        y_true:   array-like de vérité terrain (0/1)
        y_pred:   array-like de prédictions (0/1)
        prot_attr: array-like de l'attribut sensible
        priv_group: valeur du groupe privilégié
        pos_label: valeur du label positif
        sample_weight: poids des échantillons (optionnel)

    Returns:
        dict de métriques
    """
    group_metrics = {}
    group_metrics["base_rate_truth"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["statistical_parity_difference"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
        pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["disparate_impact_ratio"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
        pos_label=pos_label, sample_weight=sample_weight
    )
    if y_pred is not None:
        group_metrics["base_rate_preds"] = base_rate(
            y_true=y_pred, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
            pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
            pos_label=pos_label, sample_weight=sample_weight
        )
        if len(set(y_pred)) > 1:
            group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
                y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
                pos_label=pos_label, sample_weight=sample_weight
            )
        else:
            group_metrics["conditional_demographic_disparity"] = None
        group_metrics["smoothed_edf"] = smoothed_edf(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
            pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
            pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["balanced_accuracy_score"] = balanced_accuracy_score(
            y_true=y_true, y_pred=y_pred, sample_weight=sample_weight
        )
    return group_metrics

print("✓ Fonction get_metrics définie.")

✓ Fonction get_metrics définie.


In [98]:
def compute_reweighing_weights(df_in, label_col, protected_col):
    """
    Calcule les poids de reweighing pour rendre le label indépendant de l'attribut sensible.

    Formule : W(S=s, Y=y) = P(Y=y) × P(S=s) / P(Y=y, S=s)
    """
    n = len(df_in)
    weights = pd.Series(1.0, index=df_in.index)

    for s_val in df_in[protected_col].unique():
        for y_val in df_in[label_col].unique():
            mask = (df_in[protected_col] == s_val) & (df_in[label_col] == y_val)
            n_sy = mask.sum()
            n_s = (df_in[protected_col] == s_val).sum()
            n_y = (df_in[label_col] == y_val).sum()

            if n_sy > 0:
                w = (n_y / n) * (n_s / n) / (n_sy / n)
                weights[mask] = w

    return weights



def apply_weights_to_csv(df_source, csv_out, weights, weights_col='WEIGHTS'):
    """Sauvegarde un CSV avec pondération sur le train uniquement et poids neutres sur valid/test."""
    df_tmp = df_source.copy()
    df_tmp[weights_col] = 1.0

    if len(weights) == len(df_tmp):
        df_tmp[weights_col] = weights.values
    else:
        train_mask = df_tmp['train_valid'] == 'train'
        if train_mask.sum() != len(weights):
            raise ValueError(
                f"Longueur des poids incompatible: {len(weights)} vs {train_mask.sum()} lignes train"
            )
        df_tmp.loc[train_mask, weights_col] = weights.values

    df_tmp.to_csv(csv_out, index=False)
    print(f"  CSV sauvegardé : {csv_out}")
    print(f"  Poids — min={df_tmp[weights_col].min():.4f}, max={df_tmp[weights_col].max():.4f}, mean={df_tmp[weights_col].mean():.4f}")
    return csv_out

print("✓ Fonctions compute_reweighing_weights et apply_weights_to_csv définies.")

✓ Fonctions compute_reweighing_weights et apply_weights_to_csv définies.


In [99]:
def prepare_predictions(preds_csv_path):
    """
    Charge le CSV de prédictions et prépare les colonnes binaires pour l'analyse.

    Returns:
        DataFrame enrichi (label_binary, preds_binary, Gender_Binary, Position_Binary, score_malade)
    """
    df_p = pd.read_csv(preds_csv_path)

    # Encodage binaire
    df_p['label_binary'] = (df_p['labels'] == 'malade').astype(int)
    df_p['preds_binary'] = (df_p['preds'] == 'malade').astype(int)
    df_p['Gender_Binary'] = (df_p['Patient Gender'] == 'M').astype(int)
    df_p['Position_Binary'] = (df_p['View Position'] == 'PA').astype(int)

    # Conversion logits → probabilités via softmax
    logit_cols = [c for c in df_p.columns if c.startswith('preds_logit')]
    if len(logit_cols) >= 2:
        logits = torch.tensor(df_p[logit_cols].values.astype(float))
        probs = torch.softmax(logits, dim=1).numpy()
        df_p['score_malade'] = probs[:, 0]
        df_p['score_sain'] = probs[:, 1]

    return df_p


def evaluate_model(df_p, split='valid', prot_attr='Gender_Binary', priv_group=1, label=''):
    """
    Évalue les métriques de fairness et performance sur un split donné.

    Returns:
        dict de métriques
    """
    df_eval = df_p[df_p['train_valid'] == split].copy()

    metrics = get_metrics(
        y_true=df_eval['label_binary'].values,
        y_pred=df_eval['preds_binary'].values,
        prot_attr=df_eval[prot_attr].values,
        priv_group=priv_group,
        pos_label=1,
    )
    metrics['method'] = label
    metrics['protected_attribute'] = prot_attr

    return metrics


def print_metrics(metrics, title=""):
    """Affiche proprement un dictionnaire de métriques."""
    if title:
        print(f"\n{'=' * 60}")
        print(title)
        print('=' * 60)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:45s} : {v:+.4f}" if 'ratio' not in k else f"  {k:45s} : {v:.4f}")


def create_aif360_dataset(df_eval, label_col='label_binary', prot_attr='Gender_Binary', score_col=None):
    """Crée un BinaryLabelDataset AIF360 à partir d'un DataFrame."""
    df_aif = df_eval[[label_col, prot_attr]].copy().reset_index(drop=True)
    dataset = BinaryLabelDataset(
        df=df_aif,
        label_names=[label_col],
        protected_attribute_names=[prot_attr],
        favorable_label=1.0,
        unfavorable_label=0.0,
    )
    if score_col is not None and score_col in df_eval.columns:
        dataset.scores = df_eval[[score_col]].values.astype(float)
    return dataset


def fit_postprocessor_on_validation(df_val, method='ROC', prot_attr='Gender_Binary'):
    """Ajuste le post-processing sur la validation."""
    dataset_true = create_aif360_dataset(df_val, 'label_binary', prot_attr, score_col='score_malade')
    dataset_pred = dataset_true.copy(deepcopy=True)
    dataset_pred.labels = df_val['preds_binary'].values.reshape(-1, 1).astype(float)
    dataset_pred.scores = df_val['score_malade'].values.reshape(-1, 1).astype(float)

    privileged_groups = [{prot_attr: 1}]
    unprivileged_groups = [{prot_attr: 0}]

    if method == 'ROC':
        postproc = RejectOptionClassification(
            unprivileged_groups=unprivileged_groups,
            privileged_groups=privileged_groups,
            low_class_thresh=0.01,
            high_class_thresh=0.99,
            num_class_thresh=100,
            num_ROC_margin=50,
            metric_name="Statistical parity difference",
            metric_ub=0.05,
            metric_lb=-0.05,
        )
        postproc = postproc.fit(dataset_true, dataset_pred)
        print(f"  ROC — Seuil optimal : {postproc.classification_threshold:.4f}")
        print(f"  ROC — Marge optimale : {postproc.ROC_margin:.4f}")
    elif method == 'CalibratedEqOdds':
        postproc = CalibratedEqOddsPostprocessing(
            privileged_groups=privileged_groups,
            unprivileged_groups=unprivileged_groups,
            cost_constraint='fnr',
            seed=42,
        )
        postproc = postproc.fit(dataset_true, dataset_pred)
    else:
        raise ValueError(f"Méthode de post-processing inconnue: {method}")

    return postproc


def apply_postprocessor_to_split(df_eval, postproc, prot_attr='Gender_Binary'):
    """Applique un post-processing déjà ajusté sur un split donné."""
    dataset_true = create_aif360_dataset(df_eval, 'label_binary', prot_attr, score_col='score_malade')
    dataset_pred = dataset_true.copy(deepcopy=True)
    dataset_pred.labels = df_eval['preds_binary'].values.reshape(-1, 1).astype(float)
    dataset_pred.scores = df_eval['score_malade'].values.reshape(-1, 1).astype(float)
    dataset_corrected = postproc.predict(dataset_pred)
    return dataset_corrected.labels[:, 0]


def evaluate_post_processing(df_val, df_eval, method='ROC', prot_attr='Gender_Binary', label=''):
    """Ajuste sur validation puis évalue sur le split d'évaluation fourni."""
    postproc = fit_postprocessor_on_validation(df_val, method=method, prot_attr=prot_attr)
    corrected_preds = apply_postprocessor_to_split(df_eval, postproc, prot_attr=prot_attr)

    metrics = get_metrics(
        y_true=df_eval['label_binary'].values,
        y_pred=corrected_preds,
        prot_attr=df_eval[prot_attr].values,
        priv_group=1,
        pos_label=1,
    )
    metrics['method'] = label
    metrics['protected_attribute'] = prot_attr

    return metrics, postproc

print("✓ Fonctions prepare_predictions, evaluate_model, print_metrics, create_aif360_dataset et post-processing définies.")

✓ Fonctions prepare_predictions, evaluate_model, print_metrics, create_aif360_dataset et post-processing définies.


## II/- Analyse de l'impact de la pondération sur les performances du modèle
### 2.1 Baseline de référence (sans pondération)

In [116]:
# Compatibilite apres refacto: recharge le module d'entrainement
import importlib
import train_classifieur as tc

importlib.reload(tc)

# Rebind explicite des objets utilises dans le notebook
ChestXRayClassifier = tc.ChestXRayClassifier
pred_classifier = tc.pred_classifier
preds_todf = tc.preds_todf
train_classifier = tc.train_classifier
transforms_valid = tc.transforms_valid

In [117]:
# === Entraînement du modèle Baseline (WEIGHTS = 1 pour tout le monde) ===

dt = datetime.now()
logdir_baseline = f"{EXPE_DIR}/baseline_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"

ckpt_path_baseline, ckpt_score_baseline = train_classifier(
    logdir=logdir_baseline,
    datadir=SPLIT_DATA_DIR,
    csv=SPLIT_CSV_PATH,
    weights_col="WEIGHTS",
    max_epochs=MAX_EPOCHS,
)

print(f"\nModèle sauvegardé : {ckpt_path_baseline}")
print(f"Meilleur score (val_loss) : {ckpt_score_baseline:.4f}")

c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\Data\metadata.csv c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/baseline_2026_04_02_16_35_21/csv_in_WEIGHTS.csv
num_workers set to : 0


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


End of training 304.7824251651764

Modèle sauvegardé : C:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log\baseline_2026_04_02_16_35_21\best-val-loss.ckpt
Meilleur score (val_loss) : 0.6767


In [114]:
# === Predictions du modele Baseline sur validation ===
BASELINE_VAL_PREDS = f"{logdir_baseline}/preds_baseline_valid.csv"

df_baseline = predict_partition(SPLIT_CSV_PATH, SPLIT_DATA_DIR, 'valid', ckpt_path_baseline, preds_col='preds')
df_baseline_val = df_baseline.copy()

df_baseline.to_csv(BASELINE_VAL_PREDS, index=False)
print(f"\nPredictions validation sauvegardees : {BASELINE_VAL_PREDS}")

num_workers set to : 0


100%|██████████| 78/78 [00:34<00:00,  2.28it/s]


Predictions validation sauvegardees : c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/baseline_2026_04_02_16_26_14/preds_baseline_valid.csv


In [115]:
# === Évaluation du modèle Baseline ===
# Balanced Accuracy globale sur la validation
ba_baseline = balanced_accuracy_score(df_baseline['label_binary'], df_baseline['preds_binary'])
ba_baseline_val = balanced_accuracy_score(df_baseline_val['label_binary'], df_baseline_val['preds_binary'])
print(f"Balanced Accuracy (validation) : {ba_baseline_val:.4f}")
print(f"Balanced Accuracy (valid)       : {ba_baseline:.4f}")

# Métriques par Genre
metrics_baseline_gender = evaluate_model(df_baseline, split='valid', prot_attr='Gender_Binary', label='Baseline')
print_metrics(metrics_baseline_gender, "Baseline — Fairness par Genre (valid)")

# Métriques par Position
metrics_baseline_pos = evaluate_model(df_baseline, split='valid', prot_attr='Position_Binary', label='Baseline (pos)')
print_metrics(metrics_baseline_pos, "Baseline — Fairness par Position (valid)")

Balanced Accuracy (validation) : 0.6365
Balanced Accuracy (valid)       : 0.6365

Baseline — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0580
  disparate_impact_ratio                        : 1.2096
  base_rate_preds                               : +0.3034
  equal_opportunity_difference                  : +0.0088
  average_odds_difference                       : +0.0382
  conditional_demographic_disparity             : +0.0050
  smoothed_edf                                  : +0.1899
  df_bias_amplification                         : +0.0533
  balanced_accuracy_score                       : +0.6365

Baseline — Fairness par Position (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.1028
  disparate_impact_ratio                        : 1.3892
  base_rate_preds                               : +0.3034
  equal_opportunity_differen

### 2.2 Reweighing par genre

In [118]:
# === Calcul des poids de Reweighing (Genre) ===
print("Poids de Reweighing par Genre :")
df_train = df_split[df_split['train_valid'] == 'train'].copy()
weights_gender = compute_reweighing_weights(df_train, 'label_binary', 'Gender_Binary')

for g in [0, 1]:
    for l in [0, 1]:
        mask = (df_train['Gender_Binary'] == g) & (df_train['label_binary'] == l)
        g_name = 'M' if g == 1 else 'F'
        l_name = 'malade' if l == 1 else 'sain'
        if mask.any():
            print(f"  Genre={g_name}, Label={l_name} : poids={weights_gender[mask].iloc[0]:.4f} (n={mask.sum()})")
        else:
            print(f"  Genre={g_name}, Label={l_name} : combinaison absente du train")

CSV_GENDER = f"{WORK_DIR}/metadata_rw_gender.csv"
apply_weights_to_csv(df_split, CSV_GENDER, weights_gender)

Poids de Reweighing par Genre :
  Genre=F, Label=sain : poids=0.9310 (n=892)
  Genre=F, Label=malade : poids=1.0958 (n=643)
  Genre=M, Label=sain : poids=1.0489 (n=1259)
  Genre=M, Label=malade : poids=0.9479 (n=1182)
  CSV sauvegardé : c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/metadata_rw_gender.csv
  Poids — min=0.9310, max=1.0958, mean=1.0000


'c:\\Users\\remim\\Documents\\Projet Fairness\\Fairness-in-AI\\expe_log/metadata_rw_gender.csv'

In [119]:
# === Entraînement avec poids Reweighing Genre ===
dt = datetime.now()
logdir_gender = f"{EXPE_DIR}/rw_gender_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"

ckpt_path_gender, ckpt_score_gender = train_classifier(
    logdir=logdir_gender,
    datadir=SPLIT_DATA_DIR,
    csv=CSV_GENDER,
    weights_col="WEIGHTS",
    max_epochs=MAX_EPOCHS,
)
print(f"\nModèle RW Genre sauvegardé : {ckpt_path_gender}")
print(f"Meilleur score (val_loss) : {ckpt_score_gender:.4f}")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/metadata_rw_gender.csv c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/rw_gender_2026_04_02_16_41_24/csv_in_WEIGHTS.csv
num_workers set to : 0
num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


End of training 290.19102239608765

Modèle RW Genre sauvegardé : C:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log\rw_gender_2026_04_02_16_41_24\best-val-loss.ckpt
Meilleur score (val_loss) : 0.6735


In [120]:
# === Predictions du modele Reweighing Genre sur validation ===
GENDER_VAL_PREDS = f"{logdir_gender}/preds_rw_gender_valid.csv"

df_gender = predict_partition(CSV_GENDER, SPLIT_DATA_DIR, 'valid', ckpt_path_gender, preds_col='preds')
df_gender_val = df_gender.copy()

df_gender.to_csv(GENDER_VAL_PREDS, index=False)
print(f"\nPredictions validation sauvegardees : {GENDER_VAL_PREDS}")

num_workers set to : 0


100%|██████████| 78/78 [00:32<00:00,  2.42it/s]


Predictions validation sauvegardees : c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/rw_gender_2026_04_02_16_41_24/preds_rw_gender_valid.csv


In [121]:
# === Évaluation du modèle Reweighing Genre ===
metrics_rw_gender = evaluate_model(df_gender, split='valid', prot_attr='Gender_Binary', label='RW Genre')
print_metrics(metrics_rw_gender, "Reweighing Genre — Fairness par Genre (valid)")

# Aussi évaluer par position (effet croisé)
metrics_rw_gender_pos = evaluate_model(df_gender, split='valid', prot_attr='Position_Binary', label='RW Genre (pos)')
print_metrics(metrics_rw_gender_pos, "Reweighing Genre — Fairness par Position (valid)")


Reweighing Genre — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0448
  disparate_impact_ratio                        : 1.0738
  base_rate_preds                               : +0.6276
  equal_opportunity_difference                  : +0.0193
  average_odds_difference                       : +0.0322
  conditional_demographic_disparity             : +0.0035
  smoothed_edf                                  : +0.1206
  df_bias_amplification                         : -0.0160
  balanced_accuracy_score                       : +0.5927

Reweighing Genre — Fairness par Position (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.1923
  disparate_impact_ratio                        : 0.7255
  base_rate_preds                               : +0.6276
  equal_opportunity_difference                  : -0.2361
  average_odds_difference          

### 2.3 Reweighing par position (PA/AP)

In [122]:
# === Calcul des poids de Reweighing (Position) ===
print("Poids de Reweighing par Position :")
df_train = df_split[df_split['train_valid'] == 'train'].copy()
weights_position = compute_reweighing_weights(df_train, 'label_binary', 'Position_Binary')

for p in [0, 1]:
    for l in [0, 1]:
        mask = (df_train['Position_Binary'] == p) & (df_train['label_binary'] == l)
        p_name = 'PA' if p == 1 else 'AP'
        l_name = 'malade' if l == 1 else 'sain'
        if mask.any():
            print(f"  Position={p_name}, Label={l_name} : poids={weights_position[mask].iloc[0]:.4f} (n={mask.sum()})")
        else:
            print(f"  Position={p_name}, Label={l_name} : combinaison absente du train")

CSV_POSITION = f"{WORK_DIR}/metadata_rw_position.csv"
apply_weights_to_csv(df_split, CSV_POSITION, weights_position)

Poids de Reweighing par Position :
  Position=AP, Label=sain : poids=1.1441 (n=732)
  Position=AP, Label=malade : poids=0.8708 (n=816)
  Position=PA, Label=sain : poids=0.9257 (n=1419)
  Position=PA, Label=malade : poids=1.1045 (n=1009)
  CSV sauvegardé : c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/metadata_rw_position.csv
  Poids — min=0.8708, max=1.1441, mean=1.0000


'c:\\Users\\remim\\Documents\\Projet Fairness\\Fairness-in-AI\\expe_log/metadata_rw_position.csv'

In [123]:
# === Entraînement avec poids Reweighing Position ===
dt = datetime.now()
logdir_pos = f"{EXPE_DIR}/rw_position_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"

ckpt_path_pos, ckpt_score_pos = train_classifier(
    logdir=logdir_pos,
    datadir=SPLIT_DATA_DIR,
    csv=CSV_POSITION,
    weights_col="WEIGHTS",
    max_epochs=MAX_EPOCHS,
)
print(f"\nModèle RW Position sauvegardé : {ckpt_path_pos}")
print(f"Meilleur score (val_loss) : {ckpt_score_pos:.4f}")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/metadata_rw_position.csv c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/rw_position_2026_04_02_16_46_52/csv_in_WEIGHTS.csv
num_workers set to : 0
num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


End of training 287.5177206993103

Modèle RW Position sauvegardé : C:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log\rw_position_2026_04_02_16_46_52\best-val-loss.ckpt
Meilleur score (val_loss) : 0.6356


In [124]:
# === Prédictions du modèle Reweighing Position sur validation ===
POS_VAL_PREDS = f"{logdir_pos}/preds_rw_position_valid.csv"

df_pos = predict_partition(CSV_POSITION, SPLIT_DATA_DIR, 'valid', ckpt_path_pos, preds_col='preds')
df_pos_val = df_pos.copy()

df_pos.to_csv(POS_VAL_PREDS, index=False)
print(f"\nPredictions validation sauvegardees : {POS_VAL_PREDS}")

num_workers set to : 0


100%|██████████| 78/78 [00:31<00:00,  2.46it/s]


Predictions validation sauvegardees : c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/rw_position_2026_04_02_16_46_52/preds_rw_position_valid.csv


In [125]:
# === Évaluation du modèle Reweighing Position ===
metrics_rw_pos = evaluate_model(df_pos, split='valid', prot_attr='Position_Binary', label='RW Position')
print_metrics(metrics_rw_pos, "Reweighing Position — Fairness par Position (valid)")

# Aussi évaluer par genre (effet croisé)
metrics_rw_pos_gender = evaluate_model(df_pos, split='valid', prot_attr='Gender_Binary', label='RW Position (genre)')
print_metrics(metrics_rw_pos_gender, "Reweighing Position — Fairness par Genre (valid)")


Reweighing Position — Fairness par Position (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.0351
  disparate_impact_ratio                        : 0.9351
  base_rate_preds                               : +0.5281
  equal_opportunity_difference                  : -0.0966
  average_odds_difference                       : -0.0488
  conditional_demographic_disparity             : -0.0079
  smoothed_edf                                  : +0.0737
  df_bias_amplification                         : +0.0193
  balanced_accuracy_score                       : +0.6641

Reweighing Position — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.1236
  disparate_impact_ratio                        : 1.2625
  base_rate_preds                               : +0.5281
  equal_opportunity_difference                  : +0.0859
  average_odds_difference    

### 2.4 Reweighing croise genre x position

In [126]:
# === Reweighing croisé: poids + entraînement + prédiction + évaluation ===
df_train = df_split[df_split['train_valid'] == 'train'].copy()

# 0=F_AP, 1=F_PA, 2=M_AP, 3=M_PA
df_train['GenderPosition_Binary'] = 2 * df_train['Gender_Binary'] + df_train['Position_Binary']
weights_cross = compute_reweighing_weights(df_train, 'label_binary', 'GenderPosition_Binary')
CSV_CROSS = f"{WORK_DIR}/metadata_rw_gender_position.csv"
apply_weights_to_csv(df_split, CSV_CROSS, weights_cross)

dt = datetime.now()
logdir_cross = f"{EXPE_DIR}/rw_gender_position_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"
ckpt_path_cross, ckpt_score_cross = train_classifier(
    logdir=logdir_cross,
    datadir=SPLIT_DATA_DIR,
    csv=CSV_CROSS,
    weights_col="WEIGHTS",
    max_epochs=MAX_EPOCHS,
)
print(f"\nModèle RW croisé sauvegardé : {ckpt_path_cross}")
print(f"Meilleur score (val_loss) : {ckpt_score_cross:.4f}")

CROSS_VAL_PREDS = f"{logdir_cross}/preds_rw_gender_position_valid.csv"

df_cross = predict_partition(CSV_CROSS, SPLIT_DATA_DIR, 'valid', ckpt_path_cross, preds_col='preds')
df_cross_val = df_cross.copy()

df_cross.to_csv(CROSS_VAL_PREDS, index=False)
print(f"\nPredictions validation sauvegardees : {CROSS_VAL_PREDS}")

metrics_rw_cross_gender = evaluate_model(df_cross, split='valid', prot_attr='Gender_Binary', label='RW Genre×Position')
metrics_rw_cross_pos = evaluate_model(df_cross, split='valid', prot_attr='Position_Binary', label='RW Genre×Position')
print_metrics(metrics_rw_cross_gender, "RW Genre×Position — Fairness par Genre (valid)")
print_metrics(metrics_rw_cross_pos, "RW Genre×Position — Fairness par Position (valid)")

  CSV sauvegardé : c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/metadata_rw_gender_position.csv
  Poids — min=0.8295, max=1.2112, mean=1.0000
c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/metadata_rw_gender_position.csv c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/rw_gender_position_2026_04_02_16_53_06/csv_in_WEIGHTS.csv
num_workers set to : 0


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


End of training 284.95888328552246

Modèle RW croisé sauvegardé : C:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log\rw_gender_position_2026_04_02_16_53_06\best-val-loss.ckpt
Meilleur score (val_loss) : 0.6255
num_workers set to : 0


100%|██████████| 78/78 [00:31<00:00,  2.46it/s]


Predictions validation sauvegardees : c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\expe_log/rw_gender_position_2026_04_02_16_53_06/preds_rw_gender_position_valid.csv

RW Genre×Position — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0247
  disparate_impact_ratio                        : 1.0684
  base_rate_preds                               : +0.3732
  equal_opportunity_difference                  : -0.0053
  average_odds_difference                       : +0.0076
  conditional_demographic_disparity             : +0.0019
  smoothed_edf                                  : +0.0661
  df_bias_amplification                         : -0.0705
  balanced_accuracy_score                       : +0.6305

RW Genre×Position — Fairness par Position (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0957
  disparate_impact_ratio    

### 2.5 Comparaison des méthodes de pondération (balanced accuracy + fairness)

In [127]:
# === Tableau comparatif Pre-processing ===
required = ['df_baseline', 'df_gender', 'df_pos', 'df_cross']
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"Variables manquantes pour la comparaison pre-processing: {missing}")

all_pre_metrics = []
for df_pred, name, prot in [
    (df_baseline, 'Baseline', 'Gender_Binary'),
    (df_baseline, 'Baseline', 'Position_Binary'),
    (df_gender, 'RW Genre', 'Gender_Binary'),
    (df_gender, 'RW Genre', 'Position_Binary'),
    (df_pos, 'RW Position', 'Gender_Binary'),
    (df_pos, 'RW Position', 'Position_Binary'),
    (df_cross, 'RW Genre×Position', 'Gender_Binary'),
    (df_cross, 'RW Genre×Position', 'Position_Binary'),
]:
    m = evaluate_model(df_pred, split='valid', prot_attr=prot, label=name)
    m['attr'] = 'Genre' if 'Gender' in prot else 'Position'
    all_pre_metrics.append(m)

df_pre = pd.DataFrame(all_pre_metrics)
cols = [
    'method', 'attr', 'balanced_accuracy_score',
    'statistical_parity_difference', 'disparate_impact_ratio',
    'equal_opportunity_difference', 'average_odds_difference'
]

print("=== Tableau comparatif — Pre-processing ===")
display(df_pre[cols].sort_values(['attr', 'method']).round(4))

=== Tableau comparatif — Pre-processing ===


,method,attr,balanced_accuracy_score,statistical_parity_difference,disparate_impact_ratio,equal_opportunity_difference,average_odds_difference
0,Baseline,Genre,0.6365,0.0580,1.2096,0.0088,0.0382
2,RW Genre,Genre,0.5927,0.0448,1.0738,0.0193,0.0322
6,RW Genre×Position,Genre,0.6305,0.0247,1.0684,-0.0053,0.0076
4,RW Position,Genre,0.6641,0.1236,1.2625,0.0859,0.1024
1,Baseline,Position,0.6365,0.1028,1.3892,0.0533,0.0917
3,RW Genre,Position,0.5927,-0.1923,0.7255,-0.2361,-0.2012
7,RW Genre×Position,Position,0.6305,0.0957,1.2842,0.0737,0.0878
5,RW Position,Position,0.6641,-0.0351,0.9351,-0.0966,-0.0488


## III/- Analyse de l'impact du post-processing
### 3.1 Post-processing sur baseline et critères de comparaison

In [128]:
def apply_post_processing(df_val, df_eval, method='ROC', prot_attr='Gender_Binary', label=''):
    """
    Ajuste une méthode de post-processing sur la validation puis l'applique sur validation.

    Args:
        df_val: DataFrame de validation avec scores et prédictions
        df_eval: DataFrame de test avec scores et prédictions
        method: 'ROC' ou 'CalibratedEqOdds'
        prot_attr: attribut sensible binaire
        label: nom de la configuration

    Returns:
        dict de metriques sur le split d'evaluation
    """
    postproc = fit_postprocessor_on_validation(df_val, method=method, prot_attr=prot_attr)
    corrected_preds = apply_postprocessor_to_split(df_eval, postproc, prot_attr=prot_attr)

    metrics = get_metrics(
        y_true=df_eval['label_binary'].values,
        y_pred=corrected_preds,
        prot_attr=df_eval[prot_attr].values,
        priv_group=1,
        pos_label=1,
    )
    metrics['method'] = label
    metrics['protected_attribute'] = prot_attr

    return metrics

print("✓ Fonction apply_post_processing définie.")

✓ Fonction apply_post_processing définie.


### 3.2 Reject Option Classification (ROC) sur baseline

In [129]:
# === ROC sur Baseline (Genre puis Position) ===
print("Application de ROC sur le modèle Baseline — attribut Genre...")
metrics_roc_baseline_gender = apply_post_processing(
    df_baseline_val,
    df_baseline,
    method='ROC',
    prot_attr='Gender_Binary',
    label='Baseline + ROC',
)
print_metrics(metrics_roc_baseline_gender, "Baseline + ROC — Fairness par Genre (valid)")

print("\nApplication de ROC sur le modèle Baseline — attribut Position...")
metrics_roc_baseline_pos = apply_post_processing(
    df_baseline_val,
    df_baseline,
    method='ROC',
    prot_attr='Position_Binary',
    label='Baseline + ROC',
)
print_metrics(metrics_roc_baseline_pos, "Baseline + ROC — Fairness par Position (valid)")

Application de ROC sur le modèle Baseline — attribut Genre...


KeyError: 'score_malade'

### 3.3 Calibrated Equalized Odds sur baseline

In [ ]:
# === Calibrated EqOdds sur Baseline (Genre puis Position) ===
print("Application de Calibrated Equalized Odds sur le modèle Baseline — attribut Genre...")
metrics_ceqo_baseline_gender = apply_post_processing(
    df_baseline_val,
    df_baseline,
    method='CalibratedEqOdds',
    prot_attr='Gender_Binary',
    label='Baseline + CalEqOdds',
)
print_metrics(metrics_ceqo_baseline_gender, "Baseline + Calibrated EqOdds — Fairness par Genre (valid)")

print("\nApplication de Calibrated Equalized Odds sur le modèle Baseline — attribut Position...")
metrics_ceqo_baseline_pos = apply_post_processing(
    df_baseline_val,
    df_baseline,
    method='CalibratedEqOdds',
    prot_attr='Position_Binary',
    label='Baseline + CalEqOdds',
)
print_metrics(metrics_ceqo_baseline_pos, "Baseline + Calibrated EqOdds — Fairness par Position (valid)")

### 3.4 Combinaisons pre-processing + post-processing

In [ ]:
# === Combinaisons pre+post sur RW Genre et RW Position ===
required = ['df_gender_val', 'df_gender', 'df_pos_val', 'df_pos']
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(
        f"Variables manquantes pour les combinaisons pre+post: {missing}. "
        "Exécuter d'abord les cellules d'entraînement et de prédiction précédentes."
    )

print("Application de ROC sur RW Genre — attribut Genre...")
metrics_roc_rw_gender_gender = apply_post_processing(
    df_gender_val,
    df_gender,
    method='ROC',
    prot_attr='Gender_Binary',
    label='RW Genre + ROC',
)
print_metrics(metrics_roc_rw_gender_gender, "RW Genre + ROC — Fairness par Genre (valid)")

print("\nApplication de ROC sur RW Genre — attribut Position...")
metrics_roc_rw_gender_pos = apply_post_processing(
    df_gender_val,
    df_gender,
    method='ROC',
    prot_attr='Position_Binary',
    label='RW Genre + ROC',
)
print_metrics(metrics_roc_rw_gender_pos, "RW Genre + ROC — Fairness par Position (valid)")

print("\nApplication de CalEqOdds sur RW Genre — attribut Genre...")
metrics_ceqo_rw_gender_gender = apply_post_processing(
    df_gender_val,
    df_gender,
    method='CalibratedEqOdds',
    prot_attr='Gender_Binary',
    label='RW Genre + CalEqOdds',
)
print_metrics(metrics_ceqo_rw_gender_gender, "RW Genre + CalEqOdds — Fairness par Genre (valid)")

print("\nApplication de CalEqOdds sur RW Genre — attribut Position...")
metrics_ceqo_rw_gender_pos = apply_post_processing(
    df_gender_val,
    df_gender,
    method='CalibratedEqOdds',
    prot_attr='Position_Binary',
    label='RW Genre + CalEqOdds',
)
print_metrics(metrics_ceqo_rw_gender_pos, "RW Genre + CalEqOdds — Fairness par Position (valid)")

print("\nApplication de ROC sur RW Position — attribut Genre...")
metrics_roc_rw_pos_gender = apply_post_processing(
    df_pos_val,
    df_pos,
    method='ROC',
    prot_attr='Gender_Binary',
    label='RW Position + ROC',
)
print_metrics(metrics_roc_rw_pos_gender, "RW Position + ROC — Fairness par Genre (valid)")

print("\nApplication de ROC sur RW Position — attribut Position...")
metrics_roc_rw_pos_pos = apply_post_processing(
    df_pos_val,
    df_pos,
    method='ROC',
    prot_attr='Position_Binary',
    label='RW Position + ROC',
)
print_metrics(metrics_roc_rw_pos_pos, "RW Position + ROC — Fairness par Position (valid)")

print("\nApplication de CalEqOdds sur RW Position — attribut Genre...")
metrics_ceqo_rw_pos_gender = apply_post_processing(
    df_pos_val,
    df_pos,
    method='CalibratedEqOdds',
    prot_attr='Gender_Binary',
    label='RW Position + CalEqOdds',
)
print_metrics(metrics_ceqo_rw_pos_gender, "RW Position + CalEqOdds — Fairness par Genre (valid)")

print("\nApplication de CalEqOdds sur RW Position — attribut Position...")
metrics_ceqo_rw_pos_pos = apply_post_processing(
    df_pos_val,
    df_pos,
    method='CalibratedEqOdds',
    prot_attr='Position_Binary',
    label='RW Position + CalEqOdds',
)
print_metrics(metrics_ceqo_rw_pos_pos, "RW Position + CalEqOdds — Fairness par Position (valid)")

### 3.5 Comparaison globale et discussion des compromis

In [ ]:
# === Tableau récapitulatif global ===
required = [
    'metrics_baseline_gender', 'metrics_baseline_pos',
    'metrics_rw_gender', 'metrics_rw_pos', 'metrics_rw_pos_gender',
    'metrics_rw_cross_gender', 'metrics_rw_cross_pos',
    'metrics_roc_baseline_gender', 'metrics_roc_baseline_pos',
    'metrics_ceqo_baseline_gender', 'metrics_ceqo_baseline_pos',
    'metrics_roc_rw_gender_gender', 'metrics_roc_rw_gender_pos',
    'metrics_ceqo_rw_gender_gender', 'metrics_ceqo_rw_gender_pos',
    'metrics_roc_rw_pos_gender', 'metrics_roc_rw_pos_pos',
    'metrics_ceqo_rw_pos_gender', 'metrics_ceqo_rw_pos_pos',
]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(
        f"Variables manquantes pour le tableau global: {missing}. "
        "Exécuter d'abord toutes les cellules d'entraînement, de post-processing et de combinaisons."
    )

all_metrics = [
    {**metrics_baseline_gender, 'method': 'Baseline', 'attr': 'Genre'},
    {**metrics_baseline_pos, 'method': 'Baseline', 'attr': 'Position'},
    {**metrics_rw_gender, 'method': 'RW Genre', 'attr': 'Genre'},
    {**metrics_rw_pos_gender, 'method': 'RW Position', 'attr': 'Genre'},
    {**metrics_rw_pos, 'method': 'RW Position', 'attr': 'Position'},
    {**metrics_rw_cross_gender, 'method': 'RW Genre×Position', 'attr': 'Genre'},
    {**metrics_rw_cross_pos, 'method': 'RW Genre×Position', 'attr': 'Position'},
    {**metrics_roc_baseline_gender, 'method': 'Baseline + ROC', 'attr': 'Genre'},
    {**metrics_roc_baseline_pos, 'method': 'Baseline + ROC', 'attr': 'Position'},
    {**metrics_ceqo_baseline_gender, 'method': 'Baseline + CalEqOdds', 'attr': 'Genre'},
    {**metrics_ceqo_baseline_pos, 'method': 'Baseline + CalEqOdds', 'attr': 'Position'},
    {**metrics_roc_rw_gender_gender, 'method': 'RW Genre + ROC', 'attr': 'Genre'},
    {**metrics_roc_rw_gender_pos, 'method': 'RW Genre + ROC', 'attr': 'Position'},
    {**metrics_ceqo_rw_gender_gender, 'method': 'RW Genre + CalEqOdds', 'attr': 'Genre'},
    {**metrics_ceqo_rw_gender_pos, 'method': 'RW Genre + CalEqOdds', 'attr': 'Position'},
    {**metrics_roc_rw_pos_gender, 'method': 'RW Position + ROC', 'attr': 'Genre'},
    {**metrics_roc_rw_pos_pos, 'method': 'RW Position + ROC', 'attr': 'Position'},
    {**metrics_ceqo_rw_pos_gender, 'method': 'RW Position + CalEqOdds', 'attr': 'Genre'},
    {**metrics_ceqo_rw_pos_pos, 'method': 'RW Position + CalEqOdds', 'attr': 'Position'},
]

df_compare = pd.DataFrame(all_metrics)
cols_display = [
    'method', 'attr', 'balanced_accuracy_score',
    'statistical_parity_difference', 'disparate_impact_ratio',
    'equal_opportunity_difference', 'average_odds_difference',
    'df_bias_amplification'
]

print("=" * 100)
print("TABLEAU RÉCAPITULATIF — Toutes les méthodes et les deux attributs sensibles")
print("=" * 100)
display(df_compare[cols_display].sort_values(['attr', 'method']).round(4))

In [ ]:
# === Visualisation Pareto du compromis performance / fairness ===
if 'df_compare' not in globals():
    raise RuntimeError("df_compare absent. Exécuter d'abord la cellule du tableau global.")

df_plot = df_compare.copy()
df_plot['aod_gap'] = df_plot['average_odds_difference'].abs()

def pareto_frontier(df_in, x_col='balanced_accuracy_score', y_col='aod_gap'):
    ordered = df_in.sort_values([x_col, y_col], ascending=[False, True])
    frontier_rows = []
    best_y = float('inf')
    for _, row in ordered.iterrows():
        if row[y_col] <= best_y:
            frontier_rows.append(row)
            best_y = row[y_col]
    return pd.DataFrame(frontier_rows)

fig = px.scatter(
    df_plot,
    x='balanced_accuracy_score',
    y='aod_gap',
    color='attr',
    hover_name='method',
    hover_data={
        'balanced_accuracy_score': ':.4f',
        'aod_gap': ':.4f',
        'disparate_impact_ratio': ':.4f',
        'equal_opportunity_difference': ':.4f',
        'attr': True,
    },
    title='Pareto: balanced accuracy vs |Average Odds Difference|',
    labels={
        'balanced_accuracy_score': 'Balanced Accuracy',
        'aod_gap': '|Average Odds Difference|',
        'attr': 'Attribut sensible',
    },
)

for attr_name, color in [('Genre', '#EF553B'), ('Position', '#636EFA')]:
    frontier = pareto_frontier(df_plot[df_plot['attr'] == attr_name])
    if not frontier.empty:
        fig.add_trace(
            go.Scatter(
                x=frontier['balanced_accuracy_score'],
                y=frontier['aod_gap'],
                mode='lines+markers',
                line=dict(color=color, dash='dash'),
                marker=dict(color=color, size=10, symbol='diamond'),
                name=f'Frontière {attr_name}',
                showlegend=True,
                text=frontier['method'],
                hovertemplate='%{text}<br>BA=%{x:.4f}<br>|AOD|=%{y:.4f}<extra></extra>',
            )
        )

fig.update_layout(height=600, template='plotly_white')
fig.show()

print('Lecture rapide :')
print('- Plus on va vers le haut-gauche, plus le compromis performance/fairness est favorable.')
print('- La meilleure méthode clinique est celle qui reste proche de la frontière de Pareto tout en gardant une BA acceptable.')

In [ ]:
# === Synthèse triée pour lecture rapide ===
if 'df_compare' not in globals():
    raise RuntimeError("df_compare absent. Exécuter d'abord la cellule 40 (tableau global).")

for attr in ['Genre', 'Position']:
    print(f"\n=== Top méthodes pour {attr} ===")
    df_attr = df_compare[df_compare['attr'] == attr].copy()
    df_attr['score_tradeoff'] = (
        df_attr['balanced_accuracy_score']
        - 0.5 * df_attr['statistical_parity_difference'].abs()
        - 0.5 * (df_attr['disparate_impact_ratio'] - 1.0).abs()
    )
    keep = ['method', 'balanced_accuracy_score', 'statistical_parity_difference', 'disparate_impact_ratio', 'score_tradeoff']
    display(df_attr[keep].sort_values('score_tradeoff', ascending=False).head(5).round(4))

## 4/- Analyse et discussion
- Méthode d'interprétation
- Lecture du compromis performance / fairness
- Ce qu'on évalue explicitement dans ce notebook
- Interprétation clinique prudente


---
---
### Analyse et Discussion

### Méthode d'interprétation

L'analyse doit être lue à partir des sorties réellement exécutées dans les cellules de comparaison globale. Les conclusions ci-dessous sont formulées de manière conditionnelle et doivent être adaptées aux chiffres effectivement observés.

### Lecture du compromis performance / équité

1. Performance: la balanced accuracy mesure l'utilité clinique brute.
2. Équité: SPD proche de 0 et DIR proche de 1 indiquent une meilleure parité entre groupes.
3. Trade-off: une méthode peut améliorer l'équité mais dégrader la performance; l'objectif est une zone de compromis acceptable.

### Ce qu'on évalue explicitement dans ce notebook

- Pre-processing:
  - Baseline
  - RW Genre
  - RW Position
  - RW Genre × Position
- Post-processing:
  - ROC
  - Calibrated EqOdds
- Combinaisons pre+post:
  - RW Genre + (ROC, CalEqOdds)
  - RW Position + (ROC, CalEqOdds)
- Deux attributs sensibles:
  - Genre
  - Position

### Interprétation clinique prudente

En contexte médical, les écarts de faux négatifs entre groupes sont critiques. Une méthode peut être préférée même si la performance baisse légèrement, si elle réduit un biais cliniquement important. Le choix final doit être fait à partir du tableau global et des graphes, et non d'une affirmation a priori.

## 5/- Conclusion

Ce notebook propose un protocole complet conforme au cadrage du projet final:
- analyse des déséquilibres,
- plusieurs pre-processing,
- plusieurs post-processing,
- combinaisons pre+post,
- comparaison multi-critères sur deux attributs sensibles.

### Conclusion à renseigner après exécution complète

La conclusion finale doit s'appuyer sur les sorties de la section 6.4:
1. Méthode la plus équilibrée pour le Genre (au vu de BA, SPD, DIR, EOD, AOD).
2. Méthode la plus équilibrée pour la Position.
3. Discussion du compromis retenu et justification clinique.

### Point de rigueur

Si toutes les cellules d'entraînement/comparaison n'ont pas été exécutées dans l'état sauvegardé, il faut éviter toute affirmation forte sur "la meilleure méthode". Le notebook est maintenant structuré pour produire ces preuves de façon reproductible.